In [2]:
import os
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip


def get_spark_session(app_name="Spark_App"):
    # Đọc cấu hình từ biến môi trường (Environment Variables)
    endpoint = os.getenv("MINIO_ENDPOINT", "http://minio:9000")
    access_key = os.getenv("MINIO_ACCESS_KEY", "minioadmin")
    secret_key = os.getenv("MINIO_SECRET_KEY", "minioadmin123")

    builder = (
        SparkSession.builder.appName(app_name)
        # Các cấu hình S3A/MinIO
        .config("spark.hadoop.fs.s3a.endpoint", endpoint)
        .config("spark.hadoop.fs.s3a.access.key", access_key)
        .config("spark.hadoop.fs.s3a.secret.key", secret_key)
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
        # Cấu hình Delta Lake
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config(
            "spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog",
        )
        .config("spark.sql.shuffle.partitions", "4")
    )

    return configure_spark_with_delta_pip(builder).getOrCreate()


In [3]:
import logging
from pyspark.sql import functions as F
from config.spark_builder import get_spark_session  # Import từ file cấu hình

# Cấu hình Logging
logging.basicConfig(
    level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s"
)
log = logging.getLogger("pipeline_silver")

# Constants
BRONZE_SRC = "s3a://lakehouse/bronze/all"
SILVER_DST = "s3a://lakehouse/silver/all"

In [5]:
def process_silver():
    spark = get_spark_session("NYC_Taxi_Silver_Transformation")

    log.info("[SILVER] Đang đọc dữ liệu từ Bronze Delta...")
    df = spark.read.format("delta").load(BRONZE_SRC)
    raw_count = df.count()

    # 1. Ép kiểu và Làm sạch (Casting & Cleaning)
    # Kết hợp cast và xử lý logic thời gian
    df_transformed = (
        df.withColumn(
            "tpep_pickup_datetime", F.col("tpep_pickup_datetime").cast("timestamp")
        )
        .withColumn(
            "tpep_dropoff_datetime", F.col("tpep_dropoff_datetime").cast("timestamp")
        )
        .withColumn("passenger_count", F.col("passenger_count").cast("integer"))
        .withColumn("trip_distance", F.col("trip_distance").cast("double"))
        .withColumn("fare_amount", F.col("fare_amount").cast("double"))
        .withColumn("PULocationID", F.col("PULocationID").cast("integer"))
        .withColumn("DOLocationID", F.col("DOLocationID").cast("integer"))
    )

    # # 2. Tính toán Feature (trip_duration)
    # df_transformed = df_transformed.withColumn(
    #     "trip_duration_min",
    #     (
    #         F.unix_timestamp("tpep_dropoff_datetime")
    #         - F.unix_timestamp("tpep_pickup_datetime")
    #     )
    #     / 60.0,
    # )

    # # 3. Lọc dữ liệu lỗi & Outliers (Filtering)
    # # Gom các điều kiện lọc vào một chỗ để dễ quản lý
    # df_filtered = df_transformed.filter(
    #     (F.col("tpep_pickup_datetime").isNotNull())
    #     & (F.col("passenger_count").between(1, 6))
    #     & (F.col("trip_distance").between(0.1, 100))
    #     & (F.col("trip_duration_min").between(1, 180))
    #     & (F.col("fare_amount") > 0)
    #     & (F.col("PULocationID").between(1, 263))
    #     & (F.col("DOLocationID").between(1, 263))
    # )

    # # 4. Thêm các cột phân tích thời gian
    # df_final = (
    #     df_filtered.withColumn("pickup_date", F.to_date("tpep_pickup_datetime"))
    #     .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
    #     .withColumn("pickup_day", F.dayofweek("tpep_pickup_datetime"))
    #     .withColumn("_layer", F.lit("silver"))
    #     .withColumn("_processed_at", F.current_timestamp())
    # )

    # # 5. Ghi dữ liệu xuống tầng Silver
    # log.info(f"[SILVER] Đang ghi {df_final.count()} dòng xuống Silver Delta...")
    # (
    #     df_final.write.format("delta")
    #     .mode("overwrite")
    #     .option("overwriteSchema", "true")
    #     .save(SILVER_DST)
    # )

    # log.info("[SILVER] Hoàn thành pipeline Silver!")
    # spark.stop()


if __name__ == "__main__":
    process_silver()

26/04/07 14:13:43 WARN Utils: Your hostname, primary resolves to a loopback address: 127.0.1.1; using 192.168.1.8 instead (on interface wlp45s0)
26/04/07 14:13:43 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/namphuong/miniconda3/envs/namphuong_env/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/namphuong/.ivy2/cache
The jars for the packages stored in: /home/namphuong/.ivy2/jars
io.delta#delta-core_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-c3482893-f52f-4099-929b-c3113d17611e;1.0
	confs: [default]
	found io.delta#delta-core_2.12;2.4.0 in central
	found io.delta#delta-storage;2.4.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
downloading https://repo1.maven.org/maven2/io/delta/delta-core_2.12/2.4.0/delta-core_2.12-2.4.0.jar ...
	[SUCCESSFUL ] io.delta#delta-core_2.12;2.4.0!delta-core_2.12.jar (517ms)
downloading https://repo1.maven.org/maven2/io/delta/delta-storage/2.4.0/delta-storage-2.4.0.jar ...
	[SUCCESSFUL ] io.delta#delta-storage;2.4.0!delta-storage.jar (128ms)
:: resolution report :: resolve 827ms :: artifacts dl 657ms
	:: modules in use:
	io.delta#delta-core_2.12;2.4.0 from central in [default]
	io.delta#delta-storage;2.4.0 from central in [default]
	org.antlr#antlr4-runti

Py4JJavaError: An error occurred while calling o47.load.
: java.lang.RuntimeException: java.lang.ClassNotFoundException: Class org.apache.hadoop.fs.s3a.S3AFileSystem not found
	at org.apache.hadoop.conf.Configuration.getClass(Configuration.java:2688)
	at org.apache.hadoop.fs.FileSystem.getFileSystemClass(FileSystem.java:3431)
	at org.apache.hadoop.fs.FileSystem.createFileSystem(FileSystem.java:3466)
	at org.apache.hadoop.fs.FileSystem.access$300(FileSystem.java:174)
	at org.apache.hadoop.fs.FileSystem$Cache.getInternal(FileSystem.java:3574)
	at org.apache.hadoop.fs.FileSystem$Cache.get(FileSystem.java:3521)
	at org.apache.hadoop.fs.FileSystem.get(FileSystem.java:540)
	at org.apache.hadoop.fs.Path.getFileSystem(Path.java:365)
	at org.apache.spark.sql.delta.DeltaTableUtils$.findDeltaTableRoot(DeltaTable.scala:177)
	at org.apache.spark.sql.delta.sources.DeltaDataSource$.parsePathIdentifier(DeltaDataSource.scala:357)
	at org.apache.spark.sql.delta.catalog.DeltaTableV2.x$1$lzycompute(DeltaT